# KPoEM SHAP 워드클라우드 시각화

- 본 코드는 KPoEM 감정 분류 모델에 대한 SHAP(SHapley Additive exPlanations) 분석 결과를 시각화하기 위한 워드클라우드 생성 코드이다.
- SHAP 분석은 `backend/poet_SHAP_batched_run.ipynb`에서 수행되었으며, 분석 과정에서 생성된 SHAP pickle(.pkl) 파일을 입력 데이터로 사용한다.
- 본 워드클라우드는 한국 근현대 시인 5인(김소월, 윤동주, 이상, 임화, 한용운)의 시 텍스트에 대해 특정 감정 분류에 기여한 주요 어휘를 시각적으로 탐색하기 위해 제작되었다.
- 각 단어의 크기는 해당 단어가 감정 예측에 기여한 SHAP value의 누적값을 기반으로 결정된다.
- 불용어 제거, 최소 길이 제한, SHAP 기여도 임계값(threshold) 적용 등의 전처리 과정을 거친 후 워드클라우드가 생성된다.

# KPoEM SHAP Word Cloud Visualization

This script visualizes token-level SHAP explanations generated from the KPoEM emotion classification model.

- Input: SHAP pickle files generated by `backend/poet_SHAP_batched_run.ipynb`
- Data source: Korean modern poetry corpus (KPoEM)
- Method: Aggregation of positive SHAP values for each token
- Output: Emotion-specific word cloud visualization
- Purpose: To identify and interpret lexical features that contribute to the model's emotion predictions.

### 필요한 라이브러리 설치 및 적용

In [ ]:
!pip install matplotlib numpy wordcloud

import os, glob, pickle
import numpy as np
from collections import defaultdict
from wordcloud import WordCloud
import matplotlib.pyplot as plt


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


# 44개 이미지 한꺼번에 추출

In [ ]:
KOTE_44 = [ #44가지 감정 카테고리 (감정분석 모델이 44개로 분류하기 때문) Index를 감정명으로 매핑하기 위한 리스트
    "불평/불만", "환영/호의", "감동/감탄", "지긋지긋", "고마움",
    "슬픔", "화남/분노", "존경", "기대감", "우쭐댐/무시함",
    "안타까움/실망", "비장함", "의심/불신", "뿌듯함", "편안/쾌적",
    "신기함/관심", "아껴주는", "부끄러움", "공포/무서움", "절망",
    "한심함", "역겨움/징그러움", "짜증", "어이없음", "없음",
    "패배/자기혐오", "귀찮음", "힘듦/지침", "즐거움/신남", "깨달음",
    "죄책감", "증오/혐오", "흐뭇함(귀여움/예쁨)", "당황/난처", "경악",
    "부담/안_내킴", "서러움", "재미없음", "불쌍함/연민", "놀람",
    "행복", "불안/걱정", "기쁨", "안심/신뢰"
]

In [ ]:
PKL_DIR = "shap/shap_han"
PATTERN = "yongun_shap_batch*.pkl"   # 너 파일명 패턴에 맞게 조정

THRESH = 0.05   # 0보다 큰 것만 누적. 필요하면 0.01~0.05로 올려서 노이즈 컷
MIN_LEN = 2    # 너무 짧은 토큰 제거(원하면 1로)

STOP_TOKENS = {"", " ", "\n", "\t", "[PAD]", "[CLS]", "[SEP]", "[UNK]"} # 불용어 토큰. 필요하면 더 추가

In [ ]:
# 토큰 정제 함수.
def clean_token(tok: str) -> str:
    if not isinstance(tok, str):
        tok = str(tok)
    # sentencepiece/roberta 표식 제거
    tok = tok.replace("▁", "").replace("Ġ", "").strip()
    return tok

### 시각화하여 이미지를 저장

In [9]:
SAVE_DIR = "./wordclouds"
os.makedirs(SAVE_DIR, exist_ok=True)

for idx, emotion in enumerate(KOTE_44):

    print(f"[{idx}] {emotion}")

    token_scores = defaultdict(float)

    files = sorted(glob.glob(os.path.join(PKL_DIR, PATTERN)))

    for fp in files:

        with open(fp, "rb") as f:
            data = pickle.load(f)

        shap_values = data["shap_values"]
        sv = shap_values[:, :, idx]

        vals = np.asarray(sv.values)
        toks = np.asarray(sv.data, dtype=object)

        n_samples = vals.shape[0]

        for i in range(n_samples):

            tokens_i = list(toks[i]) if not isinstance(toks[i], str) else [toks[i]]
            vals_i = vals[i]

            m = min(len(tokens_i), len(vals_i))

            for tok, v in zip(tokens_i[:m], vals_i[:m]):

                if v <= THRESH:
                    continue

                tok = clean_token(tok)

                if (not tok) or (tok in STOP_TOKENS) or (len(tok) < MIN_LEN):
                    continue

                token_scores[tok] += float(v)

    if len(token_scores) == 0:
        print(f"skip: {emotion}")
        continue

    wc = WordCloud(
        font_path="/System/Library/Fonts/AppleSDGothicNeo.ttc",
        background_color="white",
        width=800,
        height=400
    )

    wc.generate_from_frequencies(token_scores)

    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")

    # 0.png ~ 43.png
    plt.savefig(
        os.path.join(SAVE_DIR, f"{idx}.png"),
        bbox_inches="tight",
        dpi=300
    )

    plt.close()

print("완료")

[0] 불평/불만
[1] 환영/호의
[2] 감동/감탄
[3] 지긋지긋
[4] 고마움
[5] 슬픔
[6] 화남/분노
[7] 존경
[8] 기대감
[9] 우쭐댐/무시함
[10] 안타까움/실망
[11] 비장함
[12] 의심/불신
[13] 뿌듯함
[14] 편안/쾌적
[15] 신기함/관심
[16] 아껴주는
[17] 부끄러움
[18] 공포/무서움
[19] 절망
[20] 한심함
[21] 역겨움/징그러움
[22] 짜증
[23] 어이없음
[24] 없음
[25] 패배/자기혐오
[26] 귀찮음
[27] 힘듦/지침
[28] 즐거움/신남
[29] 깨달음
[30] 죄책감
[31] 증오/혐오
[32] 흐뭇함(귀여움/예쁨)
[33] 당황/난처
[34] 경악
[35] 부담/안_내킴
[36] 서러움
[37] 재미없음
[38] 불쌍함/연민
[39] 놀람
[40] 행복
[41] 불안/걱정
[42] 기쁨
[43] 안심/신뢰
완료
